# RT-DETR-L: Tuned Fold 4 — Kaggle Submission

## Overview

This notebook generates a Kaggle submission using the best-performing individual checkpoint from our five-fold RT-DETR-L cross-validation experiments.

## Methodology

- **Architecture:** RT-DETR-L
- **Checkpoint:** Tuned Fold 4 (`best.pt`)
- **Input resolution:** 960px
- **Confidence threshold:** 0.01
- **Maximum detections:** 500
- **Number of test images:** 3,811
- **Inference device:** Tesla T4 GPU

The model predicts bounding boxes for each test image. Predictions are converted into the required `class confidence x y width height` format using pixel coordinates from the original images.

## Results

| Metric | Value |
|---|---:|
| Kaggle Public LB | **0.2869715** |
| Inference time | 9.92 minutes |
| Test images processed | 3,811 |
| Output file | `submission.csv` |

This single-model submission was used as a reference before combining the five RT-DETR-L checkpoints with Weighted Boxes Fusion.

## Reproducibility

The notebook uses `ultralytics==8.4.202` and the Fold 4 checkpoint produced by `RT-DETR-L Tuned 5-Fold CV`.

The trained weights and test images are loaded from Kaggle Inputs and are not stored in the GitHub repository.

No additional model training is performed.

In [1]:
from pathlib import Path

ROOT = Path("/kaggle/input")

models = sorted(ROOT.rglob("best.pt"))
csv_files = sorted(ROOT.rglob("test_images.csv"))

print("TRAINED MODELS:")
for p in models:
    print(p, f"({p.stat().st_size / 1024**2:.1f} MB)")

print("\nTEST DATA:")
for p in csv_files:
    print(p)

print("\nCHECK:")
print("Best checkpoints:", len(models))
print("Test CSV:", len(csv_files))

if len(models) >= 5 and csv_files:
    print("\nREADY FOR TUNED INFERENCE!")
else:
    print("\nSome files are missing — check Add Input.")

TRAINED MODELS:
/kaggle/input/notebooks/vikhimiak/rt-detr-l-tuned-5-fold-cv/rtdetr_tuned/cv_runs/tuned_cv_fold0/weights/best.pt (63.3 MB)
/kaggle/input/notebooks/vikhimiak/rt-detr-l-tuned-5-fold-cv/rtdetr_tuned/cv_runs/tuned_cv_fold1/weights/best.pt (63.3 MB)
/kaggle/input/notebooks/vikhimiak/rt-detr-l-tuned-5-fold-cv/rtdetr_tuned/cv_runs/tuned_cv_fold2/weights/best.pt (63.3 MB)
/kaggle/input/notebooks/vikhimiak/rt-detr-l-tuned-5-fold-cv/rtdetr_tuned/cv_runs/tuned_cv_fold3/weights/best.pt (63.3 MB)
/kaggle/input/notebooks/vikhimiak/rt-detr-l-tuned-5-fold-cv/rtdetr_tuned/cv_runs/tuned_cv_fold4/weights/best.pt (63.3 MB)

TEST DATA:
/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026/test_images.csv

CHECK:
Best checkpoints: 5
Test CSV: 1

READY FOR TUNED INFERENCE!


In [2]:

# ============================================================
# RT-DETR-L TUNED FOLD 4 — KAGGLE SUBMISSION
# Inference only. No training.
# ============================================================

import sys
import subprocess
import time
from pathlib import Path

import torch
import pandas as pd

try:
    import ultralytics
    if ultralytics.__version__ != "8.4.202":
        raise ImportError("Version mismatch")
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", "ultralytics==8.4.202"
    ])

from ultralytics import RTDETR

# 1. Paths
COMP = Path(
    "/kaggle/input/competitions/"
    "multiple-object-video-detection-hw-2-ucu-ai-2026"
)

MODEL_PATH = Path(
    "/kaggle/input/notebooks/vikhimiak/"
    "rt-detr-l-tuned-5-fold-cv/"
    "rtdetr_tuned/cv_runs/tuned_cv_fold4/weights/best.pt"
)

OUTPUT = Path("/kaggle/working/submission.csv")

assert COMP.is_dir(), "Competition dataset missing"
assert MODEL_PATH.is_file(), "Tuned Fold 4 checkpoint missing"
assert torch.cuda.is_available(), "Enable GPU T4"

# 2. Load test data
test_df = pd.read_csv(
    COMP / "test_images.csv",
    dtype={"image_id": str, "file_name": str}
)

assert len(test_df) == 3811
assert test_df["image_id"].is_unique
assert {"image_id", "file_name", "width", "height"}.issubset(
    test_df.columns
)

image_paths = [
    COMP / name
    for name in test_df["file_name"]
]

missing = [str(p) for p in image_paths if not p.is_file()]
assert not missing, f"Missing images: {missing[:5]}"

# 3. Load model
print("=" * 65)
print("RT-DETR-L TUNED FOLD 4 — KAGGLE INFERENCE")
print("=" * 65)
print("GPU:", torch.cuda.get_device_name(0))
print("Images:", len(test_df))
print("Checkpoint:", MODEL_PATH)

model = RTDETR(str(MODEL_PATH))

expected_names = [
    "person", "bicycle", "car", "van",
    "truck", "tricycle", "bus", "motor"
]

actual_names = [
    model.names[i] for i in range(8)
]

assert actual_names == expected_names, (
    f"Class mapping mismatch: {actual_names}"
)

print("Classes:", actual_names)
print("\nStarting inference...")

# 4. Inference
start = time.perf_counter()
rows = []

for i, row in enumerate(test_df.itertuples(index=False)):

    img_path = COMP / row.file_name

    result = model.predict(
        source=str(img_path),
        imgsz=960,
        conf=0.01,
        iou=0.7,
        max_det=500,
        device=0,
        verbose=False
    )[0]

    height, width = result.orig_shape

    assert width == int(row.width)
    assert height == int(row.height)

    tokens = []

    if result.boxes is not None:

        boxes = result.boxes.xyxy.cpu().tolist()
        classes = result.boxes.cls.cpu().tolist()
        scores = result.boxes.conf.cpu().tolist()

        for xyxy, cls, score in zip(
            boxes, classes, scores
        ):

            x1, y1, x2, y2 = xyxy

            x1 = max(0.0, min(float(x1), width))
            x2 = max(0.0, min(float(x2), width))
            y1 = max(0.0, min(float(y1), height))
            y2 = max(0.0, min(float(y2), height))

            box_w = x2 - x1
            box_h = y2 - y1

            if box_w <= 0 or box_h <= 0:
                continue

            assert 0 <= int(cls) < 8
            assert 0 <= score <= 1

            tokens.extend([
                str(int(cls)),
                f"{score:.6f}",
                f"{x1:.2f}",
                f"{y1:.2f}",
                f"{box_w:.2f}",
                f"{box_h:.2f}"
            ])

    rows.append({
        "image_id": row.image_id,
        "PredictionString": " ".join(tokens)
    })

    if (i + 1) % 200 == 0 or i + 1 == len(test_df):
        elapsed = (time.perf_counter() - start) / 60
        print(
            f"Processed {i + 1}/{len(test_df)} "
            f"| Elapsed: {elapsed:.1f} min",
            flush=True
        )

# 5. Validate submission
submission = pd.DataFrame(
    rows,
    columns=["image_id", "PredictionString"]
)

assert len(submission) == 3811
assert submission["image_id"].is_unique
assert submission["image_id"].tolist() == (
    test_df["image_id"].tolist()
)

total_detections = 0
empty_images = 0

for pred in submission["PredictionString"]:
    if not pred:
        empty_images += 1
        continue

    parts = pred.split()
    assert len(parts) % 6 == 0

    n = len(parts) // 6
    assert n <= 500

    total_detections += n

# 6. Save
temporary = OUTPUT.with_suffix(".csv.tmp")
submission.to_csv(temporary, index=False)
temporary.replace(OUTPUT)

assert OUTPUT.is_file()

print("\n" + "=" * 65)
print("RT-DETR-L TUNED FOLD 4 — SUBMISSION READY")
print("=" * 65)
print("Rows:", len(submission))
print("Total detections:", total_detections)
print("Empty images:", empty_images)
print("File size:", round(OUTPUT.stat().st_size / 1024**2, 2), "MB")
print("Time:", round((time.perf_counter() - start) / 60, 2), "min")
print("Saved to:", OUTPUT)

print("\nALL VALIDATION CHECKS PASSED!")
print("READY FOR KAGGLE SUBMISSION!")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
RT-DETR-L TUNED FOLD 4 — KAGGLE INFERENCE
GPU: Tesla T4
Images: 3811
Checkpoint: /kaggle/input/notebooks/vikhimiak/rt-detr-l-tuned-5-fold-cv/rtdetr_tuned/cv_runs/tuned_cv_fold4/weights/best.pt
Classes: ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']

Starting inference...
Processed 200/3811 | Elapsed: 0.7 min
Processed 400/3811 | Elapsed: 1.2 min
Processed 600/3811 | Elapsed: 1.8 min
Processed 800/3811 | Elapsed: 2.3 min
Processed 1000/3811 | Elapsed: 2.8 min
Processed 1200/3811 | Elapsed: 3.3 min
Processed 1400/3811 | Elapsed: 3.7 min
Processed 1600/3811 | Elapsed: 4.2 min
Processed 1800/3811 | Elapsed: 4.7 min
Processed 2000/3811 | Elapsed: 5.3 min
Processed 2200/3811 | El